# KNOWLY 分析ノートブック③　解答の記録(CSV)を分析する

ノート②で作った解答の記録(`data/log_personal.csv`)を、pandas で読み込んで、中身を調べます。
**本物のデータが取れたときも、この手順で分析できます。** データの読み込み、集計、グラフの作り方の練習です。

**前提**:ノート②を実行して、`data` フォルダに `log_personal.csv` と `log_personal_retry.csv` ができていること。

**注意**:このデータは、仮のルールで作った架空のデータです。ここで見つかる傾向は、そのルールの結果です。
本物の中学生のデータで同じ分析をして、はじめて意味のある発見になります。

**このノートで見ること**

1. データの中身を確認する
2. 日ごとの正答率
3. 生徒ごと・問題ごとの正答率
4. 解答にかかった秒数と、正解・不正解の関係
5. 前回から何日あけたかと、正答率(データから見た「忘れ方」)
6. 出題回数の偏り(放置されている問題はないか)
7. 改良の前後を比べる(「再挑戦枠」を入れたら、放置は減ったか)

## 1. データを読み込む

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# グラフで日本語を表示するための設定(Windowsに入っているフォントを順に探す)
plt.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "MS Gothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

CSV_PATH = Path("data/log_personal.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError("data/log_personal.csv が見つかりません。先にノート②を実行してください。")

# CSV を読み込む(answered_at は、日時として読み込む)
df = pd.read_csv(CSV_PATH, parse_dates=["answered_at"])

# 学習を始めてから何日目か、という列を足す
day0 = df["answered_at"].dt.normalize().min()
df["day"] = (df["answered_at"].dt.normalize() - day0).dt.days

df.head()

## 2. データの中身を確認する

分析の前に、まず「どんなデータか」を確認します。行数、人数、期間、欠けている値がないかを見ます。

In [ ]:
print("行数(解答の数):", len(df))
print("生徒の人数    :", df["student_id"].nunique())
print("問題の数      :", df["question_id"].nunique())
print("期間          :", df["answered_at"].min(), "〜", df["answered_at"].max())
print()
print("欠けている値の数(0なら欠けなし):")
print(df.isna().sum())

df[["correct", "response_sec", "attempt_no"]].describe().round(2)

## 3. 日ごとの正答率

学習が進むと、正答率はどう変わるでしょうか。1日ごとの正答率を見ます。
(最初の数日は、まだ解いていない問題が多く、勘で当てる確率の約20%に近くなります。)

In [ ]:
print(f"全体の正答率: {df['correct'].mean():.1%}")

by_day = df.groupby("day")["correct"].mean() * 100

plt.figure(figsize=(8, 3.8))
plt.plot(by_day.index, by_day.values, marker="o", markersize=3)
plt.axhline(20, color="gray", linestyle="--", linewidth=0.8)
plt.text(by_day.index.max(), 21, "勘で当たる確率(20%)", ha="right", fontsize=9)
plt.xlabel("学習を始めてから何日目か")
plt.ylabel("正答率(%)")
plt.title("日ごとの正答率")
plt.tight_layout()
plt.show()

## 4. 生徒ごと・問題ごとの正答率

`groupby` で、生徒ごと・問題ごとに集計します。

In [ ]:
by_student = df.groupby("student_id").agg(
    解答数=("correct", "size"),
    正答率=("correct", "mean"),
    平均秒数=("response_sec", "mean"),
)
by_student["正答率"] = by_student["正答率"] * 100

plt.figure(figsize=(9, 3.8))
plt.bar(by_student.index, by_student["正答率"], color="tab:blue", alpha=0.7)
plt.xticks(rotation=90, fontsize=8)
plt.ylabel("正答率(%)")
plt.title("生徒ごとの正答率")
plt.tight_layout()
plt.show()

print("正答率が高い生徒 上位3人と、低い生徒 下位3人")
by_student.sort_values("正答率", ascending=False).iloc[[0, 1, 2, -3, -2, -1]].round(1)

In [ ]:
by_question = df.groupby("question_id").agg(
    解答数=("correct", "size"),
    正答率=("correct", "mean"),
)
by_question["正答率"] = by_question["正答率"] * 100

print("正答率が低い問題(難しい問題) 上位5問")
display(by_question.sort_values("正答率").head(5).round(1))
print("正答率が高い問題(易しい問題) 上位5問")
display(by_question.sort_values("正答率", ascending=False).head(5).round(1))

## 5. 解答にかかった秒数と、正解・不正解

「すぐ答えられた問題は、正解しやすい」でしょうか。正解と不正解で、解答にかかった秒数を比べます。

In [ ]:
df["結果"] = df["correct"].map({1: "正解", 0: "不正解"})
print(df.groupby("結果")["response_sec"].agg(["count", "mean", "median"]).round(1))

plt.figure(figsize=(5, 3.8))
plt.boxplot(
    [df.loc[df["結果"] == "不正解", "response_sec"], df.loc[df["結果"] == "正解", "response_sec"]],
    tick_labels=["不正解", "正解"],
)
plt.ylabel("解答にかかった秒数")
plt.title("正解・不正解と解答時間")
plt.tight_layout()
plt.show()

## 6. 前回から何日あけたかと、正答率

同じ生徒が、同じ問題を解くときに、**前回から何日あいていたか**を計算して、正答率を見ます。
「間があくほど忘れて、正答率が下がる」という忘却曲線が、データから見えるでしょうか。

**注意**:このデータは、出し方に偏りがあります。「忘れる直前」の問題を選んで出しているので、間があいた問題は、もともと覚えの強い問題が多くなります。
ふつうの忘却曲線とは、見え方が変わります。

In [ ]:
# 同じ生徒・同じ問題を、時間順に並べて、前回の日を調べる
df = df.sort_values(["student_id", "question_id", "answered_at"])
df["prev_day"] = df.groupby(["student_id", "question_id"])["day"].shift(1)
df["gap"] = df["day"] - df["prev_day"]            # 前回から何日あいたか

rep = df[df["gap"].notna()].copy()                # 2回目以降の解答だけ
rep["間隔"] = pd.cut(
    rep["gap"],
    bins=[0, 1, 2, 4, 7, 14, 100],
    labels=["1日", "2日", "3〜4日", "5〜7日", "8〜14日", "15日以上"],
)
gap_df = rep.groupby("間隔", observed=True).agg(件数=("correct", "size"), 正答率=("correct", "mean"))
gap_df["正答率"] = gap_df["正答率"] * 100

plt.figure(figsize=(7, 3.8))
bars = plt.bar(gap_df.index.astype(str), gap_df["正答率"], color="tab:orange", alpha=0.7)
for b, n in zip(bars, gap_df["件数"]):
    plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 1, f"{n}件", ha="center", fontsize=8)
plt.ylim(0, 100)
plt.xlabel("前回から何日あいたか")
plt.ylabel("正答率(%)")
plt.title("間隔と正答率(棒の上は、その間隔の解答の件数)")
plt.tight_layout()
plt.show()

gap_df.round(1)

## 7. 出題回数の偏り(放置されている問題はないか)

ノート②で、「直前」方式は、一部の問題を放置しやすいことが分かりました。
このデータで、1人が1つの問題を何回解いたかを数えて、偏りを確認します。

In [ ]:
# 行=生徒、列=問題、値=解いた回数 の表を作る
counts = df.groupby(["student_id", "question_id"]).size().unstack(fill_value=0)
flat = counts.values.flatten()

print(f"1つの問題を解いた回数の平均: {flat.mean():.1f}回   最小: {flat.min()}回   最大: {flat.max()}回")
print(f"2回以下しか解かなかった問題の割合(放置率): {(flat <= 2).mean():.1%}")

plt.figure(figsize=(7, 3.8))
plt.hist(flat, bins=range(0, flat.max() + 2), color="tab:green", alpha=0.7, align="left")
plt.axvline(2.5, color="black", linestyle="--", linewidth=0.8)
plt.text(2.7, plt.ylim()[1] * 0.9, "2回以下は、放置", fontsize=9)
plt.xlabel("1人が1つの問題を解いた回数")
plt.ylabel("(生徒, 問題)の組の数")
plt.title("問題ごとの出題回数の分布")
plt.tight_layout()
plt.show()

In [ ]:
# 解いた回数の多さ別に、正答率を比べる
pair = df.groupby(["student_id", "question_id"]).agg(回数=("correct", "size"), 正答率=("correct", "mean"))
pair["回数の区分"] = pd.cut(pair["回数"], bins=[0, 2, 5, 9, 100], labels=["1〜2回", "3〜5回", "6〜9回", "10回以上"])
by_count = pair.groupby("回数の区分", observed=True).agg(組の数=("回数", "size"), 平均正答率=("正答率", "mean"))
by_count["平均正答率"] = by_count["平均正答率"] * 100
by_count.round(1)

## 8. 改良の前後を比べる

ノート②で、放置を減らすために、**間違えた問題をもう一度出す枠(再挑戦枠)** を入れた方式を作りました。
その記録(`log_personal_retry.csv`)を読み込んで、いままで分析してきた「直前」方式の記録と、同じ見方で比べます。

(どちらも、乱数 42 の1回分の記録です。1回分なので、細かい差は、偶然の影響もあります。
また、2つは `target`(忘れかけとみなす確率)の設定が違います。「直前」は 0.8、「直前+再挑戦」は 0.99 です(ノート②で、それぞれ選んだ値)。
放置率の数字は、`target` が高いほど大きくなるので、そのまま比べないでください。**見るのは、放置された問題の中身です。**)

In [ ]:
RETRY_PATH = Path("data/log_personal_retry.csv")
if not RETRY_PATH.exists():
    raise FileNotFoundError("data/log_personal_retry.csv が見つかりません。ノート②を最新版で実行してください。")

df_retry = pd.read_csv(RETRY_PATH, parse_dates=["answered_at"])


def summarize(d):
    """1つの記録について、放置の様子をまとめる"""
    counts = d.groupby(["student_id", "question_id"]).size()           # (生徒, 問題)ごとの解答回数
    pair_acc = d.groupby(["student_id", "question_id"])["correct"].mean()
    all_counts = counts.unstack(fill_value=0).values.flatten()
    neglected = counts[counts <= 2].index                              # 2回以下しか解かなかった組
    return {
        "解答数": len(d),
        "全体の正答率(%)": d["correct"].mean() * 100,
        "放置率(%)": (all_counts <= 2).mean() * 100,
        "放置された問題の正答率(%)": pair_acc[neglected].mean() * 100,
        "1つの問題を解いた回数の平均": all_counts.mean(),
        "いちばん多く解いた回数": all_counts.max(),
    }


compare = pd.DataFrame({"直前": summarize(df), "直前+再挑戦": summarize(df_retry)})
compare.round(1)

In [ ]:
# 出題回数の分布を、2つ並べる
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, (name, d, color) in zip(axes, [("直前", df, "tab:orange"), ("直前+再挑戦", df_retry, "tab:red")]):
    c = d.groupby(["student_id", "question_id"]).size().unstack(fill_value=0).values.flatten()
    ax.hist(c, bins=range(0, 30), color=color, alpha=0.7, align="left")
    ax.axvline(2.5, color="black", linestyle="--", linewidth=0.8)
    ax.set_xlabel("1人が1つの問題を解いた回数")
    ax.set_title(f"{name}(放置率 {(c <= 2).mean():.1%})")
axes[0].set_ylabel("(生徒, 問題)の組の数")
plt.tight_layout()
plt.show()

## 9. まとめと、次にやること

**私が動かしたときの結果**(ノート②の乱数 42 のデータ。細かい数字は、少しずれることがあります)

- 解答は6000件(20人 × 300問)。欠けている値はありませんでした。全体の正答率は約51%です。
- **生徒ごとの差**:正答率は、高い生徒で約59%、低い生徒で約36%と、かなり開きがあります。
- **解答時間**:正解のときは平均約8.6秒、不正解のときは約10.4秒でした。「迷うと間違えやすい」傾向が、データに出ています。
  (このデータは、仮のルールでそう作ってあります。本物のデータでも同じになるかは、確かめる必要があります。)
- **間隔と正答率**:「間があくほど、正答率が下がる」という、きれいな右下がりにはなりませんでした。
  「忘れる直前」の問題を選んで出しているので、間があいた問題は、もともと覚えの強いものが多いためです。
  15日以上あいた解答は19件しかなく、件数が少ないので、この区間は参考程度にしてください。
  **出し方に偏りがあるデータから、忘れ方をそのまま読み取ってはいけない**、という大事な教訓です。
- **放置された問題**:放置率は約17.5%でした。そして、1〜2回しか解かなかった組の**平均正答率は約9%**と、極端に低くなっています。
  つまり、放置された問題は、「出したのに、ほとんど解けなかった問題」です。
  一度間違えると、「正解できる確率」の予測が下がり、「忘れる直前の問題から出す」方式では、選ばれにくくなるためです。
  **苦手な問題ほど、放置されやすい**ことが、データで確認できました。

- **改良の前後(ノート②の再挑戦枠)**:放置率は、「直前」が約17.5%、「直前+再挑戦」が約22.8%でした(`target` が違うので、この数字は、そのままでは比べられません)。
  大きく変わったのは、**放置された問題の中身**です。放置された問題の正答率が、**約9%から約50%** になりました。
  つまり、「苦手なのに放置されていた」問題が、再挑戦枠で拾われるようになり、いま放置されているのは、「すでに解けている問題」が中心になった、ということです。
  全体の正答率も、約51%から約55%に上がっています。
  (これは、「間違えた問題を出す」というしくみから予想できる結果です。本物のデータでも同じになるかは、確かめる必要があります。)

**読み方のポイント**

- 点数が高い方式でも、**中身(どの問題が、どう扱われたか)を見ないと、弱点は見えません**。
  ノート②では「放置率」という数字だけでしたが、このノートで、放置されたのが苦手な問題だと分かりました。
- データの集め方に偏りがあると、見かけの関係が変わります。分析のときは、「どう集めたデータか」を、いつも確認します。

**次にやること**

- [ ] 本物の中学生のデータが取れたら、このノートの手順で、同じ分析をする
- [ ] 「次に正解できるか」を予測するモデルを、scikit-learn で作ってみる